# Lesson 3: Sentence Window Retrieval (句窗检索)

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import importlib
import utils
importlib.reload(utils)

import os
import openai

In [3]:
from llama_index.core import SimpleDirectoryReader  # 导入 LlamaIndex 的全能文件读取器

# 实例化读取器并加载数据
# input_files: 接收一个列表，指定你要加载的特定文件路径
documents = SimpleDirectoryReader(
    input_files=["./eBook-How-to-Build-a-Career-in-AI.pdf"]
).load_data()  # 这里的 load_data() 会解析 PDF，将其拆解为一系列 Document 对象

In [4]:
print(type(documents), "\n")
print(len(documents), "\n")
print(type(documents[0]))
print(documents[0])

<class 'list'> 

41 

<class 'llama_index.core.schema.Document'>
Doc ID: a92d7af3-a0bf-4e67-a1bc-27fc887f2dde
Text: PAGE 1 Founder, DeepLearning.AI Collected Insights from Andrew
Ng How to  Build Your Career in AI A Simple Guide


In [5]:
from llama_index.core import Document

document = Document(text="\n\n".join([doc.text for doc in documents]))

## Window-sentence retrieval setup (句窗检索设置)

In [6]:
from llama_index.core.node_parser import SentenceWindowNodeParser

# create the sentence window node parser w/ default settings
# 创建句子窗口节点解析器
node_parser = SentenceWindowNodeParser.from_defaults(
    # 使用 from_defaults() 创建解析器实例，并设置关键参数。
    
    # window_size=3: 定义“窗口”的大小。
    # 对于任何被检索到的“核心句子”，其上下文将包括这个核心句子
    # 以及其“前面 3 个句子”和“后面 3 个句子”，总共 7 个句子。
    window_size=1,
    # window_metadata_key="window": 指定存储“大窗口文本”（即核心句子+周围句子）的元数据键名。
    # 在检索时，LLM会使用这个键名下的完整窗口文本作为上下文。
    window_metadata_key="window",
    # original_text_metadata_key="original_text": 指定存储“核心句子”（即用于向量化和检索的文本块）的元数据键名。
    # 核心句子本身会被向量化并用于检索，但它的大窗口文本（"window"）会被提供给LLM。
    original_text_metadata_key="original_text",
)

In [7]:
text = "hello. how are you? I am fine!  "

nodes = node_parser.get_nodes_from_documents([Document(text=text)])

In [8]:
print([x.text for x in nodes])

['hello. ', 'how are you? ', 'I am fine!  ']


In [9]:
print(nodes[1].metadata["window"])

hello.  how are you?  I am fine!  


In [10]:
text = "hello. foo bar. cat dog. mouse"

nodes = node_parser.get_nodes_from_documents([Document(text=text)])

In [11]:
print([x.text for x in nodes])

['hello. ', 'foo bar. ', 'cat dog. ', 'mouse']


In [12]:
print(nodes[0].metadata["window"])

hello.  foo bar. 


### 📝 LlamaIndex 核心笔记：SentenceWindowNodeParser 深度解析

#### 1. 工作流程 (Workflow)
`SentenceWindowNodeParser` 并非简单的文本切分，而是一个“语义增强”过程：

* **第一步：文本切分 (Sentences Splitting)**
    解析器基于标点符号（如 `.`, `?`, `!`）将输入文本识别为独立的句子。
    *例如：* `hello.` / `how are you?` / `I am fine!`
* **第二步：节点转化 (Node Creation)**
    为每个句子创建一个 `TextNode`。每个 Node 除了包含原始短句外，还会注入包含上下文的 **Metadata（元数据）**。



#### 2. 运行结果解析 (Instance Analysis)
假设输入文本为 `"hello. how are you? I am fine!"`，配置 `window_size=3`：

| 节点索引 | 核心内容 (`text`) | 窗口元数据 (`metadata["window"]`) |
| :--- | :--- | :--- |
| **Node 0** | "hello." | "hello. how are you? I am fine!" |
| **Node 1** | "how are you?" | "hello. how are you? I am fine!" |
| **Node 2** | "I am fine!" | "hello. how are you? I am fine!" |

#### 3. 核心机制：为什么 Window 内容相同？
因为配置的 `window_size=3` 代表“向上寻找 3 句，向下寻找 3 句”。
* 在总句数只有 3 句的情况下，每一句在寻找窗口时都会触及文本边界。
* 因此，在这个特定例子中，每个节点的窗口都恰好覆盖了全文。

#### 4. 🚀 为什么在 RAG 中这样做？
这种策略完美解决了 **“语义完整性”** 与 **“检索精度”** 的矛盾：

1.  **检索阶段 (Retrieval)**：向量数据库只对比简短的句子。句子越短，语义越聚焦，相似度计算越精准。
2.  **合成阶段 (Synthesis)**：当 Node 被检索到后，LlamaIndex 会自动提取 `metadata["window"]` 里的完整背景信息。
3.  **结果**：LLM 接收到的是包含完整上下文的文本，有效防止了因切片过小导致的“断章取义”问题，显著提升了回答的质量和逻辑性。

### Building the index (建立索引)

In [13]:
from llama_index.llms.openai_like import OpenAILike

# OpenAILike: LlamaIndex 中用于连接与 OpenAI API 兼容的服务的类
# 这里用于连接阿里云的通义千问 (DashScope) 服务
#    配置了 API Key、基础 URL (api_base) 和模型 (qwen-max)。
#    设置了较低的 temperature=0.1，以获得更确定性的回答。
#    设置了较大的 context_window=128000。
llm = OpenAILike(
    api_key=utils.get_dashscope_api_key(),
    api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen-max",
    temperature=0.1,
    context_window=128000,
    is_chat_model=True,
    is_function_calling_model=False,
)

In [14]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Settings: LlamaIndex 的全局配置对象
# 将全局的语言模型设置为配置好的 OpenAILike 实例
Settings.llm = llm
model_real_path = os.path.expanduser("~/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a"
)
# 将全局的嵌入模型设置为本地加载的 BGE-Small 模型，用于将文本内容转化为向量
Settings.embed_model = HuggingFaceEmbedding(
        # model_name=LOCAL_BGE_PATH,
        model_name=model_real_path,
        # 如果您的机器有 GPU，建议设置 device="cuda"
        device="cpu", 
        # 连不了外网记得这个标志要设置为True，不然虽然本地有了还会掉huggingface获取包信息检验
        local_files_only=True,
    )
# 设置节点解析器,节点解析器负责将原始文档（Document）分割成更小的、可管理的块，这些块被称为节点（Nodes）。
Settings.node_parser = node_parser


2026-04-25 15:31:54,522 - INFO - Load pretrained SentenceTransformer: /Users/a1-6/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a


In [15]:
from llama_index.core import VectorStoreIndex

# VectorStoreIndex.from_documents: 这是 LlamaIndex 中创建向量索引的核心步骤。
# 它使用全局配置的 embed_model，将前面合并的 document 切分（默认按块）并嵌入成向量，存储在一个向量存储中。
sentence_index = VectorStoreIndex.from_documents(
    [document]
)

In [16]:
# 将新创建的索引的存储上下文（包括向量存储、文档存储等）持久化到本地文件系统。
# 这样下次运行程序时就可以直接加载，无需重新创建。
sentence_index.storage_context.persist(persist_dir="./sentence_index")


In [17]:
# This block of code is optional to check
# if an index file exist, then it will load it
# if not, it will rebuild it

import os
from llama_index.core import VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index.core import load_index_from_storage

sentence_context = StorageContext.from_defaults()

# 检查本地目录 ./sentence_index 是否存在。
if not os.path.exists("./sentence_index"):
    # 如果目录不存在，则首次创建索引。
    sentence_index = VectorStoreIndex.from_documents(
        [document], service_context=sentence_context
    )

    # 并将创建的索引保存到本地。
    sentence_index.storage_context.persist(persist_dir="./sentence_index")
else:
    # 如果目录存在，则直接从本地加载已持久化的索引。
    sentence_index = load_index_from_storage(
        # 从指定的持久化目录创建 StorageContext。
        StorageContext.from_defaults(persist_dir="./sentence_index"),
        # 加载时需要提供相同的服务上下文（主要是 LLM 和嵌入模型）。
        service_context=sentence_context
    )

2026-04-25 15:32:08,747 - INFO - Loading all indices.


### Building the postprocessor (构建后处理器)

In [18]:
from llama_index.core.indices.postprocessor import MetadataReplacementPostProcessor

# 实例化元数据替换后处理器。
postproc = MetadataReplacementPostProcessor(
    # 核心配置：指定目标元数据键为 "window"。
    # 这个处理器会在检索后被调用，它会查找节点的元数据中 key="window" 的值，
    # 并用这个值（即完整的窗口文本）替换节点的 text 属性（即最初的小核心句子）。
    target_metadata_key="window"
)

In [19]:
from llama_index.core.schema import NodeWithScore
from copy import deepcopy

# 假设 nodes 是检索器返回的原始 Node 列表。
# 为这些节点创建一个 NodeWithScore 列表，score 设置为 1.0 (分数在此处不重要)。
scored_nodes = [NodeWithScore(node=x, score=1.0) for x in nodes]
# 复制原始节点列表，用于后续对比。
nodes_old = [deepcopy(n) for n in nodes]

In [20]:
# 打印第二个节点（索引 1）的原始文本。
# 原始文本应该是短小的“核心句子”。
nodes_old[1].text
# 注意：在实际代码中，打印 nodes_old[1].text 的结果没有被显示，但这是用于对比的参考点。

'foo bar. '

In [21]:
# 对带分数的节点列表应用后处理器。
# 后处理器执行替换操作：将节点的 text 替换为其元数据中 "window" 键对应的值。
replaced_nodes = postproc.postprocess_nodes(scored_nodes)

In [22]:
print(replaced_nodes[1].text)

hello.  foo bar.  cat dog. 


### Adding a reranker (添加重排名)

In [23]:
from llama_index.core.indices.postprocessor import SentenceTransformerRerank

# BAAI/bge-reranker-base
# link: https://huggingface.co/BAAI/bge-reranker-base
# --- 1. 定义重排序器（Reranker） ---
# 实例化 SentenceTransformerRerank 后处理器。
# 它使用一个基于 Sentence Transformer 架构的预训练模型来进行重排序。
reranker_base_path = os.path.expanduser("~/Desktop/AIAgent/models/models--BAAI--bge-reranker-base/snapshots/2cfc18c9415c912f9d8155881c133215df768a70")
rerank = SentenceTransformerRerank(
    # top_n=2: 指定重排序后最终保留的前 N 个节点。
    # 无论输入多少节点，最终只会输出得分最高的 2 个节点。
    top_n=2, 
    # model="BAAI/bge-reranker-base": 指定用于计算重排序得分的模型。
    # BAAI/bge-reranker-base 是一个流行的、用于提高语义匹配度的重排序模型
    model=reranker_base_path
)

In [24]:
# --- 2. 演示重排序器的效果 ---
from llama_index.core import QueryBundle
from llama_index.core.schema import TextNode, NodeWithScore

# 创建一个查询捆绑对象，包含用户查询。
query = QueryBundle("I want a dog.")

# 创建一个模拟的初始检索结果列表（NodeWithScore 列表）。
scored_nodes = [
    NodeWithScore(node=TextNode(text="This is a cat"), score=0.6),
    NodeWithScore(node=TextNode(text="This is a dog"), score=0.4),
]

In [25]:
# 对这些节点应用重排序后处理器。
# 重排序器会根据查询（query）和节点文本，重新计算分数。
reranked_nodes = rerank.postprocess_nodes(
    scored_nodes, query_bundle=query
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [26]:
# 打印重排序后的节点列表（文本和新的分数）。
# 预期结果：包含"dog"的节点在新分数上会更高，并排在第一位（因为 top_n=2）。
print([(x.text, x.score) for x in reranked_nodes])

[('This is a dog', 0.9182743), ('This is a cat', 0.0014040726)]


### Runing the query engine (运行查询引擎)

In [27]:
# --- 3. 将重排序器集成到查询引擎中 ---

# 假设 sentence_index 是前面创建的句子窗口索引。
# 将索引转化为查询引擎。
sentence_window_engine = sentence_index.as_query_engine(
    # similarity_top_k=6: 第一阶段（向量检索）检索前 6 个节点。
    # 这 6 个节点会被传入后处理器。
    similarity_top_k=6, 
    # node_postprocessors: 定义节点后处理器链。
    # 1. postproc (MetadataReplacementPostProcessor)：先将检索到的核心句子替换为完整的“窗口文本”。
    # 2. rerank (SentenceTransformerRerank)：再对替换后的（包含完整上下文的）节点进行重排序。
    node_postprocessors=[postproc, rerank]
)

In [28]:
# --- 4. 执行查询与结果展示 ---

# 使用配置好的查询引擎执行查询。
window_response = sentence_window_engine.query(
    "What are the keys to building a career in AI?"
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-25 15:32:33,637 - INFO - Retrying request to /chat/completions in 0.482709 seconds
2026-04-25 15:32:36,743 - INFO - Retrying request to /chat/completions in 0.927861 seconds
2026-04-25 15:32:41,095 - INFO - Retrying request to /chat/completions in 1.829619 seconds
2026-04-25 15:32:45,545 - WARNING - Retrying llama_index.llms.openai.base.OpenAI._chat in 0.7169594233684488 seconds as it raised APIConnectionError: Connection error..
2026-04-25 15:32:49,696 - INFO - Retrying request to /chat/completions in 0.418291 seconds
2026-04-25 15:32:52,954 - INFO - Retrying request to /chat/completions in 0.837340 seconds
2026-04-25 15:32:56,376 - INFO - Retrying request to /chat/completions in 1.591929 seconds
2026-04-25 15:33:01,881 - WARNING - Retrying llama_index.llms.openai.base.OpenAI._chat in 0.7372124395347934 seconds as it raised APIConnectionError: Connection error..
2026-04-25 15:33:05,522 - INFO - Retrying request to /chat/completions in 0.381680 seconds
2026-04-25 15:33:09,479 -

In [29]:
from llama_index.core.response.notebook_utils import display_response

# 在 Jupyter/Colab 环境中以美观的方式显示最终的 LLM 响应。
display_response(window_response)

**`Final Response:`** To build a career in AI, it's important to focus on three main steps: acquiring foundational technical skills, gaining practical experience through projects, and securing a job. Additionally, being part of a supportive community can greatly enhance your journey. Each of these steps plays a crucial role in establishing and advancing your career in the field of AI.

## Putting it all Together (联合起来)

In [30]:
import os
from llama_index.core import ServiceContext, VectorStoreIndex, StorageContext
from llama_index.core.node_parser import SentenceWindowNodeParser
from llama_index.core.indices.postprocessor import MetadataReplacementPostProcessor
from llama_index.core.indices.postprocessor import SentenceTransformerRerank
from llama_index.core import load_index_from_storage
from llama_index.core.settings import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

def build_sentence_window_index(
    documents,
    llm,
    embed_model,
    sentence_window_size=3,
    save_dir="sentence_index",
):
    # create the sentence window node parser w/ default settings
    node_parser = SentenceWindowNodeParser.from_defaults(
        window_size=sentence_window_size,
        window_metadata_key="window",
        original_text_metadata_key="original_text",
    )
    Settings.llm = llm
    Settings.embed_model  = HuggingFaceEmbedding(
        # model_name=LOCAL_BGE_PATH,
        model_name=embed_model,
        # 如果您的机器有 GPU，建议设置 device="cuda"
        device="mps", 
        # 连不了外网记得这个标志要设置为True，不然虽然本地有了还会掉huggingface获取包信息检验
        local_files_only=True,
    )
    Settings.node_parser = node_parser
    if not os.path.exists(save_dir):
        sentence_index = VectorStoreIndex.from_documents(
            documents
        )
        sentence_index.storage_context.persist(persist_dir=save_dir)
    else:
        sentence_index = load_index_from_storage(
            StorageContext.from_defaults(persist_dir=save_dir)
        )

    return sentence_index


def get_sentence_window_query_engine(
    sentence_index, similarity_top_k=6, rerank_top_n=2
):
    # define postprocessors
    postproc = MetadataReplacementPostProcessor(target_metadata_key="window")
    reranker_base_path = os.path.expanduser("~/Desktop/AIAgent/models/models--BAAI--bge-reranker-base/snapshots/2cfc18c9415c912f9d8155881c133215df768a70")
    rerank = SentenceTransformerRerank(
        top_n=rerank_top_n, model=reranker_base_path
    )

    sentence_window_engine = sentence_index.as_query_engine(
        similarity_top_k=similarity_top_k, node_postprocessors=[postproc, rerank]
    )
    return sentence_window_engine

In [31]:
model_real_path = os.path.expanduser("~/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a")
index = build_sentence_window_index(
    [document],
    llm=llm,
    embed_model=model_real_path,
    save_dir="./sentence_index",
)


2026-04-25 15:33:25,507 - INFO - Load pretrained SentenceTransformer: /Users/a1-6/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a
2026-04-25 15:33:26,207 - INFO - Loading all indices.


In [32]:
query_engine = get_sentence_window_query_engine(index, similarity_top_k=6)


## TruLens Evaluation (TruLens评价)

In [33]:
eval_questions = []
with open('generated_questions.text', 'r') as file:
    for line in file:
        # Remove newline character and convert to integer
        item = line.strip()
        eval_questions.append(item)

In [34]:
from trulens.core import Tru 

def run_evals(eval_questions, tru_recorder, query_engine):
    for question in eval_questions:
        with tru_recorder as recording:
            response = query_engine.query(question)

In [35]:
from utils import get_prebuilt_trulens_recorder

from trulens.core import Tru 

Tru().reset_database()

2026-04-25 15:33:33,508 - INFO - Context impl SQLiteImpl.
2026-04-25 15:33:33,508 - INFO - Will assume non-transactional DDL.
2026-04-25 15:33:33,527 - INFO - ✅ OpenTelemetry exporter set: NoneType
2026-04-25 15:33:33,565 - INFO - ✅ Added new TrulensOtelSpanProcessor
2026-04-25 15:33:33,566 - INFO - Instrumenting AsyncBatches.create for cost tracking
2026-04-25 15:33:33,567 - INFO - Instrumenting AsyncCompletions.create for cost tracking
2026-04-25 15:33:33,567 - INFO - Instrumenting AsyncContainers.create for cost tracking
2026-04-25 15:33:33,567 - INFO - Instrumenting AsyncEmbeddings.create for cost tracking
2026-04-25 15:33:33,568 - INFO - Instrumenting AsyncEvals.create for cost tracking
2026-04-25 15:33:33,568 - INFO - Instrumenting AsyncFiles.create for cost tracking
2026-04-25 15:33:33,569 - INFO - Instrumenting AsyncModerations.create for cost tracking
2026-04-25 15:33:33,569 - INFO - Instrumenting AsyncUploads.create for cost tracking
2026-04-25 15:33:33,569 - INFO - Instrumen

🦑 Initialized with db url sqlite:///default.sqlite .
🛑 Secret keys may be written to the database. See the `database_redact_keys` option of `TruSession` to prevent this.
✅ experimental Feature.OTEL_TRACING enabled.
🔒 experimental Feature.OTEL_TRACING is enabled and cannot be changed.


Updating app_name and app_version in apps table: 0it [00:00, ?it/s]
Updating app_id in records table: 0it [00:00, ?it/s]
Updating app_json in apps table: 0it [00:00, ?it/s]


### Sentence window size = 1

In [36]:
model_real_path = os.path.expanduser("~/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a"
)
sentence_index_1 = build_sentence_window_index(
    documents,
    llm=llm,
    embed_model=model_real_path,
    sentence_window_size=1,
    save_dir="sentence_index_1",
)

In [37]:
sentence_window_engine_1 = get_sentence_window_query_engine(
    sentence_index_1
)

In [38]:
tru_recorder_1 = get_prebuilt_trulens_recorder(
    sentence_window_engine_1,
    app_id='sentence window engine 1'
)

instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.embeddings.multi_modal_base.MultiModalEmbedding'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.base.embeddings.base.BaseEmbedding'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.schema.TransformComponent'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.schema.BaseComponent'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'pydantic.main.BaseModel'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base

In [39]:
run_evals(eval_questions, tru_recorder_1, sentence_window_engine_1)

##### 🚀 RAG 实验执行：自动化评估流程

这一行代码 `run_evals(...)` 标志着从配置阶段进入了**执行阶段**。它会遍历所有的测试问题，并实时监控 RAG 的每一个动作。

#### 1. 参数拆解 (Parameters)
* **`eval_questions`**: 你的题库。包含之前从 `eval_questions.txt` 中读取的所有测试问题。
* **`tru_recorder_1`**: 你的“行车记录仪”。它内置了评估指标（Feedbacks）和应用标识（App ID），负责全程录像并打分。
* **`sentence_window_engine_1`**: 你的“待考选手”。这是你构建的带有“句窗检索”能力的查询引擎。

#### 2. 内部执行逻辑 (Inner Logic)
当这行代码运行时，系统会发生以下连锁反应：

1.  **批量循环**：依次取出 `eval_questions` 中的每一个问题。
2.  **上下文捕获**：`tru_recorder_1` 拦截 `sentence_window_engine_1` 的内部动作。它会记录：
    * **Input**: 原始问题是什么？
    * **Retrieval**: 检索到了哪些单句？扩充后的窗口上下文（Window Context）是什么？
    * **Output**: LLM 最终给出的答案是什么？
3.  **触发评估 (Feedback Loop)**：
    * 记录仪会将“问题、上下文、答案”这三元组发送给之前定义的 `LiteLLM` 考官。
    * 考官根据诚实度（Groundedness）、相关性等指标实时计算分值。

#### 3. 运行结果 (Outcome)
运行结束后：
* **数据库持久化**：所有的调用链路（Traces）和评分（Scores）都会被写入 TruLens 的本地数据库。
* **仪表盘同步**：你现在可以回到 `tru.run_dashboard()` 启动的页面。点击对应的 `App ID`，就能看到一张详尽的报表，告诉你这个版本的 RAG 在各个维度上的平均分是多少。



---
**💡 笔记心得：**
`run_evals` 的核心意义在于**“可重复性”**。通过同一套 `eval_questions`，你可以不断调整 `window_size` 或更换 LLM，然后运行这个函数来观察分数的涨落，从而科学地优化你的 AI 应用。

In [40]:
Tru().run_dashboard()

Starting dashboard ...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Accordion(children=(VBox(children=(VBox(children=(Label(value='STDOUT'), Output())), VBox(children=(Label(valu…

Dashboard started at http://localhost:55616 .


<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>

### Note about the dataset of questions（关于问题数据集的说明）
- Since this evaluation process takes a long time to run, the following file `generated_questions.text` contains one question (the one mentioned in the lecture video).
- 由于这个评估过程运行时间较长，下面的文件 generated_questions.text 仅包含一个问题（也就是课程视频中提到的那一个问题）。
- If you would like to explore other possible questions, feel free to explore the file directory by clicking on the "Jupyter" logo at the top right of this notebook. You'll see the following `.text` files:
- 如果你想探索其他可能的问题，可以点击笔记本右上角的 “Jupyter” 标志，进入文件目录，你会看到以下 .text 文件：

> - `generated_questions_01_05.text`
> - `generated_questions_06_10.text`
> - `generated_questions_11_15.text`
> - `generated_questions_16_20.text`
> - `generated_questions_21_24.text`

Note that running an evaluation on more than one question can take some time, so we recommend choosing one of these files (with 5 questions each) to run and explore the results.
请注意，对多个问题进行评估会耗费较多时间，因此我们建议你先从这些文件中选择其中一个（每个文件包含 5 个问题）来运行评估并查看结果。

- For evaluating a personal project, an eval set of 20 is reasonable.
- 如果是评估个人项目，20 个问题的评估集已经是一个合理的规模。
- For evaluating business applications, you may need a set of 100+ in order to cover all the use cases thoroughly.
- 如果是评估业务级应用，你可能需要 100 个以上的问题，才能充分覆盖所有使用场景
- Note that since API calls can sometimes fail, you may occasionally see null responses, and would want to re-run your evaluations.  So running your evaluations in smaller batches can also help you save time and cost by only re-running the evaluation on the batches with issues.
- 另外，由于 API 调用有时可能会失败，你可能会看到空的响应结果，这时需要重新运行评估。因此，将评估拆分成较小的批次，可以帮助你节省时间和成本，只需重新运行出现问题的那部分批次即可。

In [41]:
eval_questions = []
with open('generated_questions.text', 'r') as file:
    for line in file:
        # Remove newline character and convert to integer
        item = line.strip()
        eval_questions.append(item)

### Sentence window size = 3

In [42]:
model_real_path = os.path.expanduser("~/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a"
)
sentence_index_3 = build_sentence_window_index(
    documents,
    llm=llm,
    embed_model=model_real_path,
    sentence_window_size=5,
    save_dir="sentence_index_5",
)
sentence_window_engine_3 = get_sentence_window_query_engine(
    sentence_index_3
)

tru_recorder_3 = get_prebuilt_trulens_recorder(
    sentence_window_engine_3,
    app_id='sentence window engine 3'
)

instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.embeddings.multi_modal_base.MultiModalEmbedding'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.base.embeddings.base.BaseEmbedding'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.schema.TransformComponent'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'llama_index.core.schema.BaseComponent'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base <class 'pydantic.main.BaseModel'>
instrumenting <class 'llama_index.embeddings.huggingface.base.HuggingFaceEmbedding'> for base

In [43]:
run_evals(eval_questions, tru_recorder_3, sentence_window_engine_3)

In [44]:
Tru().run_dashboard()

Starting dashboard ...
Dashboard already running at path:   Local URL: http://localhost:55616



<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>

In [45]:
Tru().stop_dashboard()